# General-purpose model retrieval controls

Use the existing working Qwen GPU environment. No package upgrade is performed.
Default 50-task pilot; set E4_REVIEW_MAX_ROWS=0 to finish all remaining tasks with unchanged settings.
For 250 queries and one bridge, 7 arms produce 1,750 planned reports. Inspect coverage, not raw line counts.
The new single-attempt log resumes after interruption, skipping both successful and recorded failed tasks.
Original/rescue outputs are not reused. All arms use the same newly pinned model revision.
Review success, failure categories, content, and input budgets before continuing the pilot.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get("EXPERIMENT4_CODE_ROOT", str(Path.cwd())))
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    raise FileNotFoundError("Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 code folder")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
os.environ.setdefault("EXPERIMENT4_ROOT", str(Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", str(CODE_ROOT.parent / "artifacts"))) / "experiment4"))
os.environ.setdefault("HF_HOME", str(CODE_ROOT.parent / ".cache" / "huggingface"))
from e4_review_controls import roots
SOURCE_ROOT, REVIEW_ROOT = roots()
print({"source_read_only": str(SOURCE_ROOT), "new_output": str(REVIEW_ROOT)})


In [ ]:
from huggingface_hub import login, get_token
from getpass import getpass
token_path = Path(os.environ.get("HF_TOKEN_FILE", ""))
token = token_path.read_text().strip() if token_path.is_file() else (os.environ.get("HF_TOKEN") or get_token())
if not token: token = getpass("Hugging Face read token: ").strip()
if any(c.isspace() for c in token): raise ValueError("Token file must contain exactly one token")
login(token=token, add_to_git_credential=False)
del token
print("Hugging Face authentication configured; token not displayed.")


In [ ]:
# Set to 1 only after inspecting notebook 18's images.
os.environ.setdefault("E4_REVIEW_IMAGES_APPROVED", "0")
from e4_review_controls import run_generation
run_generation("qwen35_4b", max_rows=int(os.environ.get("E4_REVIEW_MAX_ROWS", "50")))
